# Notebook note

The paper figures are rebuilt from saved results by `python scripts/make_figures.py --all`. This notebook is the place to rerun the matched-budget MLP control or change the schedules.


# Matched-budget MLP controls

This control asks whether front-loading dropout helps because of where the dropout is placed, rather than simply because some layers see a larger local rate. It compares constant dropout with front-loaded schedules and higher-budget constant baselines.


In [ ]:
from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "dropout_mft").exists():
        sys.path.insert(0, str(_root / "src"))
        break

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm
import contextlib
import warnings
import pickle

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = (device == 'cuda')
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Settings

N_SIMULATIONS = 25

# Dropout parameters
H_BAR = 0.1
H_MAX = 0.2

# Network geometry
DEPTH_MULTIPLIER = 2
SIGMA_B_SQ = 0.02
SIGMA_W_SQ = 1.98

# Which schedules to test
SCHEDULES = ['none', 'constant', 'reverse_step', 'big_step', 'double', 'triple']

# Architecture
WIDTH = 256
EPOCHS = 75
LEARNING_RATE = 1e-4
BATCH_SIZE = 100

# Learning rate schedule
USE_LR_SCHEDULE = True
LR_MIN = 1e-7

# Evaluation
EVAL_EVERY = 1

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000

In [ ]:
# Plot style

plt.style.use('seaborn-v0_8-whitegrid')

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'lines.linewidth': 1.5,
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'mathtext.fontset': 'cm',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
})

from dropout_mft.schedules import DISPLAY_NAMES
from dropout_mft.style import SCHEDULE_COLORS, schedule_color

COLORS = SCHEDULE_COLORS

def get_color(sched):
    return schedule_color(sched)

def get_display_name(sched):
    return DISPLAY_NAMES.get(sched, sched)

In [ ]:
# Dropout schedules

def get_dropout_schedule(schedule_type, depth, h_bar, h_max=None, seed=None, epoch=0):
    """
    Generate layer-wise dropout rates for different scheduling strategies.

    All schedules are designed to have mean dropout rate = h_bar,
    but differ in their spatial distribution across layers.
    """
    if schedule_type == 'none':
        return [0.0] * depth

    if schedule_type == 'constant':
        return [h_bar] * depth

    if schedule_type == 'double':
        return [h_bar * 2] * depth

    if schedule_type == 'triple':
        return [h_bar * 3] * depth

    if schedule_type == 'linear':
        # Increasing: 0 → 2*h_bar
        if depth == 1:
            return [h_bar]
        return [2 * h_bar * ell / (depth - 1) for ell in range(depth)]

    if schedule_type == 'reverse_linear':
        # Decreasing: 2*h_bar → 0
        if depth == 1:
            return [h_bar]
        return [2 * h_bar * (depth - 1 - ell) / (depth - 1) for ell in range(depth)]

    if schedule_type == 'big_step':
        # Dropout concentrated in first 1/3 of layers
        if h_max is None or h_max < h_bar:
            raise ValueError("big_step requires h_max >= h_bar")
        f = 1 / 3
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [h_adj] * n_drop + [0.0] * (depth - n_drop)

    if schedule_type == 'step':
        # Dropout only in second half (late)
        if h_max is None or h_max < h_bar:
            raise ValueError("step requires h_max >= h_bar")
        f = h_bar / h_max
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [0.0] * (depth - n_drop) + [h_adj] * n_drop

    if schedule_type == 'reverse_step':
        # Dropout only in first half (early)
        if h_max is None or h_max < h_bar:
            raise ValueError("reverse_step requires h_max >= h_bar")
        f = h_bar / h_max
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [h_adj] * n_drop + [0.0] * (depth - n_drop)

    if schedule_type == 'v_shape':
        # High at edges, zero in middle
        if depth == 1:
            return [h_bar]
        return [2 * h_bar * abs(2 * ell / (depth - 1) - 1) for ell in range(depth)]

    if schedule_type == 'inverse_v':
        # Zero at edges, high in middle
        if depth == 1:
            return [h_bar]
        return [2 * h_bar * (1 - abs(2 * ell / (depth - 1) - 1)) for ell in range(depth)]

    raise ValueError(f"Unknown schedule: {schedule_type}")


def compute_effective_xi(h_layers, schedule_type=None, h_bar_target=None):
    """
    Compute effective correlation length for ReLU networks.

    The correlation length controls how far signals propagate through
    the network before being attenuated. Longer ξ means better gradient flow.
    """
    kappa = 2 * np.sqrt(2) / (3 * np.pi)
    alpha = (3 / 2) * kappa ** (2 / 3)
    L = len(h_layers)

    # Damage accumulation (nonlinear in h due to ReLU geometry)
    damage = sum(h ** (1 / 3) for h in h_layers if h > 0) / L

    xi_inv = alpha * damage
    xi_eff = 1 / xi_inv if xi_inv > 0 else float('inf')

    return {
        'xi_eff': xi_eff,
        'h_bar': np.mean(h_layers),
        'h_max': max(h_layers) if h_layers else 0,
        'n_dropout': sum(1 for h in h_layers if h > 0)
    }

In [ ]:
# Pick depth from the correlation length

kappa = 2 * np.sqrt(2) / (3 * np.pi)
alpha = (3 / 2) * kappa ** (2 / 3)
xi_constant = 1 / (alpha * H_BAR ** (1 / 3))

DEPTH = int(DEPTH_MULTIPLIER * xi_constant)

print("Configuration")
print("=" * 50)
print(f"σ_w² = {SIGMA_W_SQ:.4f}  (He init = 2.0)")
print(f"σ_b² = {SIGMA_B_SQ:.4f}")
print(f"h̄ = {H_BAR:.4f}")
print()
print(f"Correlation length (constant): ξ = {xi_constant:.2f}")
print(f"Network depth: L = {DEPTH}  ({DEPTH_MULTIPLIER}× ξ)")
print(f"Ratio L/ξ = {DEPTH / xi_constant:.2f}")
print()
print(f"Regime check:")
print(f"  L/N = {DEPTH / WIDTH:.4f}")
print(f"  h̄/(L/N) = {H_BAR / (DEPTH / WIDTH):.1f}")
print(f"  {'✓ dropout dominates' if H_BAR > DEPTH / WIDTH else '✗ finite-width effects'}")

In [ ]:
# Schedule diagnostics

theory = {}

print(f"{'Schedule':<15} {'h̄':>8} {'h_max':>8} {'#drop':>6} {'ξ_eff':>8} {'L/ξ':>6}")
print("-" * 55)

for sched in SCHEDULES:
    h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, H_MAX, seed=0)
    stats = compute_effective_xi(h_layers, schedule_type=sched, h_bar_target=H_BAR)
    theory[sched] = {'h_layers': h_layers, **stats}

    xi_str = f"{stats['xi_eff']:.2f}" if stats['xi_eff'] < 1e6 else "∞"
    ratio = DEPTH / stats['xi_eff'] if stats['xi_eff'] < 1e6 else 0

    print(f"{sched:<15} {stats['h_bar']:>8.4f} {stats['h_max']:>8.4f} "
          f"{stats['n_dropout']:>6} {xi_str:>8} {ratio:>6.2f}")

In [ ]:
# Draw the schedules

fig, ax = plt.subplots(figsize=(6, 3.5))
layers = np.arange(1, DEPTH + 1)

for sched in SCHEDULES:
    h = np.array(theory[sched]['h_layers'])
    xi = theory[sched]['xi_eff']
    xi_str = f"{xi:.1f}" if xi < 1e6 else r"$\infty$"

    if 'step' in sched:
        ax.step(layers, h, where='mid', color=get_color(sched), lw=2,
                label=f"{get_display_name(sched)} ($\\xi$={xi_str})")
    else:
        ax.plot(layers, h, '-o', color=get_color(sched), lw=1.5, ms=3,
                markeredgecolor='white', markeredgewidth=0.3,
                label=f"{get_display_name(sched)} ($\\xi$={xi_str})")

ax.axhline(H_BAR, color='0.5', ls=':', alpha=0.8, lw=1, label=f'Mean h̄={H_BAR}')
ax.set_xlabel('Layer $\\ell$')
ax.set_ylabel('Dropout probability $h_\\ell$')
ax.set_title('Dropout schedules')
ax.legend(fontsize=7, loc='upper right')
ax.set_xlim(0, DEPTH + 1)
ax.set_ylim(-0.01, max(H_MAX, 3 * H_BAR) * 1.15)

plt.tight_layout()
plt.savefig('dropout_schedules.pdf', format='pdf')
plt.show()

In [ ]:
# Model

class CriticalReLUNet(nn.Module):
    """
    Fully-connected ReLU network with layer-wise dropout.

    Initialized near criticality with specified weight and bias variances.
    """

    def __init__(self, input_dim, hidden_dim, output_dim, h_layers,
                 sigma_w_sq=2.0, sigma_b_sq=0.0,
                 schedule_type=None, h_bar=None, seed=None):
        super().__init__()
        self.depth = len(h_layers)
        self.sigma_w_sq = sigma_w_sq
        self.sigma_b_sq = sigma_b_sq

        # Layers
        layers = [nn.Linear(input_dim, hidden_dim)]
        for _ in range(self.depth - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
        layers.append(nn.Linear(hidden_dim, output_dim))

        self.layers = nn.ModuleList(layers)
        self.dropouts = nn.ModuleList([nn.Dropout(p=h) for h in h_layers])
        self._init_critical()

    def _init_critical(self):
        """Initialize weights for signal propagation near criticality."""
        for layer in self.layers:
            fan_in = layer.weight.shape[1]
            std_w = np.sqrt(self.sigma_w_sq / fan_in)
            nn.init.normal_(layer.weight, mean=0.0, std=std_w)

            if self.sigma_b_sq > 0:
                nn.init.normal_(layer.bias, mean=0.0, std=np.sqrt(self.sigma_b_sq))
            else:
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = torch.relu(layer(x))
            x = self.dropouts[i](x)
        return self.layers[-1](x)

In [ ]:
# Training code

def autocast_context():
    if not USE_AMP:
        return contextlib.nullcontext()
    return autocast(enabled=True, dtype=AMP_DTYPE)


def iterate_batches(x, y, batch_size, shuffle=True):
    n = x.shape[0]
    idx = torch.randperm(n, device=x.device) if shuffle else torch.arange(n, device=x.device)
    for i in range(0, n, batch_size):
        yield x[idx[i:i + batch_size]], y[idx[i:i + batch_size]]


def load_cifar10(train_size=None, test_size=None, seed=0):
    """Load CIFAR-10 with optional subsampling."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

    train_set = datasets.CIFAR10('./data', train=True, download=True)
    test_set = datasets.CIFAR10('./data', train=False, download=True)

    rng = np.random.RandomState(seed)

    if train_size is None or train_size >= len(train_set.data):
        tr_idx = np.arange(len(train_set.data))
    else:
        tr_idx = rng.choice(len(train_set.data), train_size, replace=False)

    if test_size is None or test_size >= len(test_set.data):
        te_idx = np.arange(len(test_set.data))
    else:
        te_idx = rng.choice(len(test_set.data), test_size, replace=False)

    print(f"Train: {len(tr_idx)} samples, Test: {len(te_idx)} samples")

    x_tr = torch.from_numpy(train_set.data[tr_idx]).permute(0, 3, 1, 2).float() / 255
    y_tr = torch.tensor(np.array(train_set.targets)[tr_idx])
    x_te = torch.from_numpy(test_set.data[te_idx]).permute(0, 3, 1, 2).float() / 255
    y_te = torch.tensor(np.array(test_set.targets)[te_idx])

    if device == 'cuda':
        x_tr, y_tr = x_tr.cuda(), y_tr.cuda()
        x_te, y_te = x_te.cuda(), y_te.cuda()
        mean, std = mean.cuda(), std.cuda()

    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    return (x_tr, y_tr), (x_te, y_te)


def train_epoch(model, data, optimizer, criterion, batch_size):
    model.train()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0

    for xb, yb in iterate_batches(x, y, batch_size):
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            out = model(xb)
            loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)

    return loss_sum / total, 100 * correct / total


@torch.no_grad()
def evaluate(model, data, criterion, batch_size):
    model.eval()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0

    for xb, yb in iterate_batches(x, y, batch_size, shuffle=False):
        with autocast_context():
            out = model(xb)
            loss = criterion(out, yb)
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)

    return loss_sum / total, 100 * correct / total

In [ ]:
# Data

print("Loading CIFAR-10...")
train_data, test_data = load_cifar10(TRAIN_SIZE, TEST_SIZE)
print(f"Train shape: {train_data[0].shape}")

In [ ]:
# Run

def run_experiments(schedules, n_sims, epochs):
    results = {}

    for sched in schedules:
        print(f"\n{'=' * 50}")
        print(f"Schedule: {get_display_name(sched)}")
        print(f"{'=' * 50}")

        h_layers = theory[sched]['h_layers']
        histories = []

        for sim in range(n_sims):
            seed = 42 + sim
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = CriticalReLUNet(
                input_dim=3072,
                hidden_dim=WIDTH,
                output_dim=10,
                h_layers=h_layers,
                sigma_w_sq=SIGMA_W_SQ,
                sigma_b_sq=SIGMA_B_SQ
            ).to(device)

            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-7)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=LR_MIN) if USE_LR_SCHEDULE else None
            criterion = nn.CrossEntropyLoss()

            hist = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
            best_test_acc = 0.0

            pbar = tqdm(range(epochs), desc=f'Sim {sim + 1}/{n_sims}', leave=True)

            for ep in pbar:
                tr_loss, tr_acc = train_epoch(model, train_data, optimizer, criterion, BATCH_SIZE)
                if scheduler:
                    scheduler.step()

                if ep % EVAL_EVERY == 0 or ep == epochs - 1:
                    te_loss, te_acc = evaluate(model, test_data, criterion, BATCH_SIZE)
                else:
                    te_loss = hist['test_loss'][-1] if hist['test_loss'] else 0
                    te_acc = hist['test_acc'][-1] if hist['test_acc'] else 0

                best_test_acc = max(best_test_acc, te_acc)

                hist['train_acc'].append(tr_acc)
                hist['test_acc'].append(te_acc)
                hist['train_loss'].append(tr_loss)
                hist['test_loss'].append(te_loss)

                pbar.set_postfix({
                    'tr_L': f'{tr_loss:.3f}',
                    'tr_A': f'{tr_acc:.1f}%',
                    'te_A': f'{te_acc:.1f}%',
                    'best': f'{best_test_acc:.1f}%'
                })

            histories.append(hist)

        # Aggregate curves across seeds curves across seeds
        results[sched] = {
            'train_acc': np.array([h['train_acc'] for h in histories]),
            'test_acc': np.array([h['test_acc'] for h in histories]),
            'train_loss': np.array([h['train_loss'] for h in histories]),
            'test_loss': np.array([h['test_loss'] for h in histories]),
        }

        best = results[sched]['test_acc'].max(axis=1)
        print(f"{get_display_name(sched)}: Best test = {best.mean():.2f}% ± {best.std():.2f}%")

    return results


results = run_experiments(SCHEDULES, N_SIMULATIONS, EPOCHS)

In [ ]:
# Save

save_data = {
    'results': results,
    'theory': theory,
    'config': {
        'SCHEDULES': SCHEDULES,
        'N_SIMULATIONS': N_SIMULATIONS,
        'EPOCHS': EPOCHS,
        'DEPTH': DEPTH,
        'WIDTH': WIDTH,
        'H_BAR': H_BAR,
        'H_MAX': H_MAX,
        'SIGMA_W_SQ': SIGMA_W_SQ,
        'SIGMA_B_SQ': SIGMA_B_SQ,
        'LEARNING_RATE': LEARNING_RATE,
    }
}

with open('dropout_experiment_results.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("Results saved to dropout_experiment_results.pkl")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Load the pickle file
with open('/content/dropout_MLP_triple.pkl', 'rb') as f:
    data = pickle.load(f)

# Pull out the saved pieces
results = data['results']
theory = data['theory']
config = data['config']

# Pull out the plotting config
SCHEDULES = config['SCHEDULES']
DEPTH = config['DEPTH']
H_BAR = config['H_BAR']

# Recreate the plotting state
# These were globals in the original run.
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({'font.size': 10, 'lines.linewidth': 1.5})


from dropout_mft.schedules import DISPLAY_NAMES
from dropout_mft.style import SCHEDULE_COLORS, schedule_color

COLORS = SCHEDULE_COLORS

def get_color(sched):
    return schedule_color(sched)

def get_display_name(sched):
    return DISPLAY_NAMES.get(sched, sched)

print("Data loaded successfully!")
print(f"Schedules found: {SCHEDULES}")

In [ ]:
# Four-panel training curves

fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))

lines, labels = [], []

for sched in SCHEDULES:
    c = get_color(sched)
    xi = theory[sched]['xi_eff']
    xi_str = f'{xi:.1f}' if xi < 1e6 else r'$\infty$'
    label = f'{get_display_name(sched)} ($\\xi$={xi_str})'

    # Training loss
    m = results[sched]['train_loss'].mean(0)[10:]
    s = results[sched]['train_loss'].std(0)[10:]
    epochs_plot = np.arange(10, 10 + len(m))
    line, = axes[0, 0].semilogy(epochs_plot, m, color=c, lw=1.5)
    axes[0, 0].fill_between(epochs_plot, np.maximum(m - s, 1e-6), m + s, color=c, alpha=0.15)
    lines.append(line)
    labels.append(label)

    # Test loss
    m = results[sched]['test_loss'].mean(0)[10:]
    s = results[sched]['test_loss'].std(0)[10:]
    axes[0, 1].semilogy(epochs_plot, m, color=c, lw=1.5)
    axes[0, 1].fill_between(epochs_plot, np.maximum(m - s, 1e-6), m + s, color=c, alpha=0.15)

    # Training accuracy
    m = results[sched]['train_acc'].mean(0)[10:]
    s = results[sched]['train_acc'].std(0)[10:]
    epochs_full = np.arange(len(m))
    axes[1, 0].plot(epochs_full, m, color=c, lw=1.5)
    axes[1, 0].fill_between(epochs_full, m - s, m + s, color=c, alpha=0.15)

    # Test accuracy
    m = results[sched]['test_acc'].mean(0)[10:]
    s = results[sched]['test_acc'].std(0)[10:]
    axes[1, 1].plot(epochs_full, m, color=c, lw=1.5)
    axes[1, 1].fill_between(epochs_full, m - s, m + s, color=c, alpha=0.15)

axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training loss')
axes[0, 0].set_title('(a) Training loss', loc='left')

axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Test loss')
axes[0, 1].set_title('(b) Test loss', loc='left')

axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Training accuracy (%)')
axes[1, 0].set_title('(c) Training accuracy', loc='left')

axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Test accuracy (%)')
axes[1, 1].set_title('(d) Test accuracy', loc='left')

fig.legend(lines, labels, loc='lower center', ncol=3, fontsize=8,
           framealpha=0.95, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig('dropout_comparison_4panel.pdf', format='pdf', bbox_inches='tight')
plt.savefig('dropout_comparison_4panel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Final accuracy distribution
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

final_accs = {sched: results[sched]['test_acc'][:, -1] for sched in SCHEDULES}

# Histogram
ax = axes[0]
all_vals = np.concatenate(list(final_accs.values()))
bins = np.linspace(all_vals.min() - 1, all_vals.max() + 1, 20)

for sched in SCHEDULES:
    c = get_color(sched)
    vals = final_accs[sched]
    mu = vals.mean()
    ax.hist(vals, bins=bins, alpha=0.4, color=c,
            label=f'{get_display_name(sched)} ($\\mu$={mu:.1f}%)',
            edgecolor='white', linewidth=0.5)
    ax.axvline(mu, color=c, linestyle='--', linewidth=1.2)

ax.set_xlabel('Final test accuracy (%)')
ax.set_ylabel('Count')
ax.set_title('(a) Distribution of final accuracy', loc='left')
ax.legend(fontsize=6, framealpha=0.95, loc='upper left')

# Boxplot
ax = axes[1]
box_data = [final_accs[sched] for sched in SCHEDULES]
box_labels = [get_display_name(sched) for sched in SCHEDULES]
bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.6)

for patch, sched in zip(bp['boxes'], SCHEDULES):
    patch.set_facecolor(get_color(sched))
    patch.set_alpha(0.6)
    patch.set_edgecolor('0.3')

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color('0.3')

ax.set_ylabel('Final test accuracy (%)')
ax.set_title('(b) Accuracy by schedule', loc='left')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('final_accuracy_distribution.pdf', format='pdf')
plt.savefig('final_accuracy_distribution.png', dpi=300)
plt.show()

In [ ]:
# Summary table

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"{'Schedule':<20} {'ξ_eff':>8} {'L/ξ':>8} {'Best Test':>18}")
print("-" * 60)

for sched in SCHEDULES:
    xi = theory[sched]['xi_eff']
    best = results[sched]['test_acc'].max(axis=1)
    xi_str = f"{xi:.2f}" if xi < 1e6 else "∞"
    ratio = f"{DEPTH / xi:.2f}" if xi < 1e6 else "0.00"

    # Standard error of the mean
    sem = best.std() / (N_SIMULATIONS ** 0.5)

    print(f"{get_display_name(sched):<20} {xi_str:>8} {ratio:>8} {best.mean():>8.2f}% ± {sem:.2f}%")

print("=" * 65)
print(f"\nConfiguration: L={DEPTH}, N={WIDTH}, σ_w²={SIGMA_W_SQ}, σ_b²={SIGMA_B_SQ}")
print(f"Dropout: h̄={H_BAR}, {N_SIMULATIONS} simulations × {EPOCHS} epochs")

In [ ]:
# =============================================================================
# Correlation length vs accuracy
# =============================================================================

fig, ax = plt.subplots(figsize=(5.5, 4))

for sched in SCHEDULES:
    xi_eff = theory[sched]['xi_eff']
    final_acc = results[sched]['test_acc'][:, -1]

    if xi_eff == float('inf') or xi_eff > 1e6:
        continue

    ax.errorbar(xi_eff, final_acc.mean(), yerr=final_acc.std(),
                fmt='o', markersize=10, capsize=4, capthick=1.5,
                color=get_color(sched), markeredgecolor='white', markeredgewidth=1,
                label=get_display_name(sched), zorder=5)

# Reference line for no-dropout baseline
if 'none' in results:
    none_acc = results['none']['test_acc'][:, -1].mean()
    none_std = results['none']['test_acc'][:, -1].std()
    ax.axhline(none_acc, color=get_color('none'), linestyle='-', alpha=0.8, lw=1.5)
    ax.axhspan(none_acc - none_std, none_acc + none_std, color=get_color('none'), alpha=0.15)
    ax.text(ax.get_xlim()[0] + 0.1, none_acc + 1,
            f'No dropout: {none_acc:.1f}%', fontsize=9, color=get_color('none'))

ax.axvline(DEPTH, color='0.5', ls=':', lw=1, alpha=0.7)
ax.text(DEPTH + 0.15, ax.get_ylim()[0] + 1, f'$L={DEPTH}$', fontsize=8, color='0.5', rotation=90, va='bottom')

ax.set_xlabel(r'Effective correlation length $\xi_{\rm eff}$')
ax.set_ylabel('Final test accuracy (%)')
ax.set_title('Correlation length vs performance')
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('xi_vs_accuracy.pdf', format='pdf', bbox_inches='tight')
plt.savefig('xi_vs_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()